# Human Face Recognition Framework Using CNN and Transfer Learning on RAF-DB Dataset
**Author:** Bhanu Prasad Mudraboina  
**Subject:** Machine Learning  
**Professor:** Raja Hashmi Ali  
**Dataset:** [RAF-DB Dataset on Kaggle](https://www.kaggle.com/datasets/shuvoalok/raf-db-dataset)  

---

This notebook implements a complete CNN-based facial expression recognition system on the Real-world Affective Faces Database (RAF-DB).  
It trains a custom CNN and compares it against two transfer learning models: **ResNet50** and **MobileNetV2**.

**7 Emotion Classes:** Neutral, Happiness, Surprise, Sadness, Anger, Disgust, Fear

## 1. Install and Import Libraries

In [ ]:
# Install required libraries (uncomment if running on Kaggle)
# !pip install tensorflow scikit-learn matplotlib seaborn -q

import os
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Output directory for figures
os.makedirs('figures', exist_ok=True)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Configuration and Hyperparameters

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────────

# Dataset root — adjust to your Kaggle input path
# On Kaggle: /kaggle/input/raf-db-dataset/DATASET/
DATA_ROOT = '/kaggle/input/raf-db-dataset/DATASET/'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
TEST_DIR  = os.path.join(DATA_ROOT, 'test')

# Image dimensions
IMG_HEIGHT = 224
IMG_WIDTH  = 224
IMG_SIZE   = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS   = 3

# Training params
BATCH_SIZE    = 32
EPOCHS_CNN    = 50       # custom CNN
EPOCHS_TL     = 30       # transfer learning (fine-tune)
LEARNING_RATE = 1e-4
VAL_SPLIT     = 0.15     # 15% of training set → validation

# Emotion classes (RAF-DB basic emotion subset)
CLASS_NAMES = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
NUM_CLASSES = len(CLASS_NAMES)

print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Image size: {IMG_SIZE}, Batch: {BATCH_SIZE}')

## 3. Dataset Exploration and Class Distribution

In [ ]:
def count_images(directory):
    """Count images per class in a directory."""
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            n = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
            counts[cls] = n
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

print('\n=== Dataset Distribution ===')
print(f'{"Class":<12} {"Train":>8} {"Test":>8} {"Total":>8}')
print('-' * 40)
for cls in sorted(train_counts.keys()):
    tr = train_counts.get(cls, 0)
    te = test_counts.get(cls, 0)
    print(f'{cls:<12} {tr:>8} {te:>8} {tr+te:>8}')
total_train = sum(train_counts.values())
total_test  = sum(test_counts.values())
print('-' * 40)
print(f'{"TOTAL":<12} {total_train:>8} {total_test:>8} {total_train+total_test:>8}')

# ─── Plot Class Distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RAF-DB Class Distribution', fontsize=15, fontweight='bold')

colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
classes = sorted(train_counts.keys())

axes[0].bar(classes, [train_counts[c] for c in classes], color=colors)
axes[0].set_title('Training Set')
axes[0].set_xlabel('Emotion Class')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate([train_counts[c] for c in classes]):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

axes[1].bar(classes, [test_counts.get(c, 0) for c in classes], color=colors)
axes[1].set_title('Test Set')
axes[1].set_xlabel('Emotion Class')
axes[1].set_ylabel('Number of Images')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate([test_counts.get(c, 0) for c in classes]):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/class_distribution.png')

## 4. Data Loading, Preprocessing, and Augmentation

In [ ]:
# ─── Training Data Generator with Augmentation ────────────────────────────────
# Augmentation helps address class imbalance and improves generalisation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,           # Normalize pixel values to [0, 1]
    validation_split=VAL_SPLIT,  # 15% held out for validation
    rotation_range=20,           # Random rotation up to 20°
    width_shift_range=0.15,      # Horizontal shift
    height_shift_range=0.15,     # Vertical shift
    shear_range=0.1,             # Shear transformation
    zoom_range=0.15,             # Random zoom
    horizontal_flip=True,        # Mirror faces (valid for expressions)
    brightness_range=[0.8, 1.2], # Brightness variation
    fill_mode='nearest'          # Fill empty pixels
)

# Test generator — only normalisation, NO augmentation
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# ─── Create Generators ────────────────────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Class index mapping
class_indices = train_generator.class_indices
idx_to_class  = {v: k for k, v in class_indices.items()}
print('Class indices:', class_indices)
print(f'\nTrain batches: {len(train_generator)} | '
      f'Val batches: {len(val_generator)} | '
      f'Test batches: {len(test_generator)}')

In [ ]:
# ─── Visualise Sample Images from Each Class ──────────────────────────────────
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle('Sample Images per Class (RAF-DB)', fontsize=14, fontweight='bold')

for col, cls in enumerate(sorted(class_indices.keys())):
    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for row in range(2):
        if imgs:
            img_path = os.path.join(cls_path, random.choice(imgs))
            img = plt.imread(img_path)
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(cls, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/sample_images.png')

## 5. Custom CNN Model Architecture

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=7):
    """
    Custom CNN architecture for facial expression recognition.
    
    Architecture:
    - 4 Convolutional blocks (Conv2D + BatchNorm + MaxPool + Dropout)
    - Global Average Pooling
    - 2 Dense layers with Dropout
    - Softmax output
    """
    model = models.Sequential(name='Custom_CNN')

    # ── Block 1: 32 filters ────────────────────────────────────────────────────
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                            input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # ── Block 2: 64 filters ────────────────────────────────────────────────────
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # ── Block 3: 128 filters ───────────────────────────────────────────────────
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.30))

    # ── Block 4: 256 filters ───────────────────────────────────────────────────
    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.30))

    # ── Classifier Head ────────────────────────────────────────────────────────
    model.add(layers.GlobalAveragePooling2D())   # Replace Flatten to reduce params
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.50))
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.40))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model


custom_cnn = build_custom_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
                               num_classes=NUM_CLASSES)
custom_cnn.summary()

In [ ]:
# ─── Compile Custom CNN ───────────────────────────────────────────────────────
custom_cnn.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ─── Callbacks ────────────────────────────────────────────────────────────────
cnn_callbacks = [
    EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        'best_custom_cnn.keras', monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    ),
    CSVLogger('figures/custom_cnn_training_log.csv')
]

print('Custom CNN compiled and ready for training.')

## 6. Train the Custom CNN

In [ ]:
print('Training Custom CNN...')
cnn_history = custom_cnn.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=cnn_callbacks,
    verbose=1
)
print('Custom CNN training complete.')

In [ ]:
def plot_training_curves(history, model_name, save_path):
    """Plot and save accuracy and loss training curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=14, fontweight='bold')

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy',      color='steelblue')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='coral', linestyle='--')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train Loss',      color='steelblue')
    axes[1].plot(history.history['val_loss'], label='Validation Loss', color='coral', linestyle='--')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')

plot_training_curves(cnn_history, 'Custom CNN', 'figures/custom_cnn_curves.png')

## 7. Transfer Learning — ResNet50

In [ ]:
def build_transfer_model(base_model_fn, model_name, input_shape, num_classes):
    """
    Build a transfer learning model using the given base architecture.
    Strategy:
      1. Load base model with ImageNet weights (frozen)
      2. Add custom classifier head
      3. Fine-tune: unfreeze top layers
    """
    base = base_model_fn(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False  # Freeze all base layers initially

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)           # Run base in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=model_name)
    return model, base


resnet_model, resnet_base = build_transfer_model(
    ResNet50,
    model_name='ResNet50_Transfer',
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    num_classes=NUM_CLASSES
)

resnet_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_resnet50.keras', monitor='val_accuracy', save_best_only=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    CSVLogger('figures/resnet50_training_log.csv')
]

print('\n=== Phase 1: ResNet50 — Frozen Base Training ===')
resnet_history_1 = resnet_model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=resnet_callbacks,
    verbose=1
)

# ── Fine-Tuning: Unfreeze top 30 layers ─────────────────────────────────────
print('\n=== Phase 2: ResNet50 — Fine-Tuning Top Layers ===')
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:
    layer.trainable = False

resnet_model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_resnet50_finetuned.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1),
]

resnet_history_2 = resnet_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=val_generator,
    callbacks=resnet_callbacks_ft,
    verbose=1
)

plot_training_curves(resnet_history_2, 'ResNet50 (Fine-Tuned)', 'figures/resnet50_curves.png')

## 8. Transfer Learning — MobileNetV2

In [ ]:
mobilenet_model, mobilenet_base = build_transfer_model(
    MobileNetV2,
    model_name='MobileNetV2_Transfer',
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    num_classes=NUM_CLASSES
)

mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_mobilenetv2.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    CSVLogger('figures/mobilenetv2_training_log.csv')
]

print('=== Phase 1: MobileNetV2 — Frozen Base Training ===')
mobilenet_history_1 = mobilenet_model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=mobilenet_callbacks,
    verbose=1
)

print('\n=== Phase 2: MobileNetV2 — Fine-Tuning Top Layers ===')
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_mobilenetv2_finetuned.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1),
]

mobilenet_history_2 = mobilenet_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=val_generator,
    callbacks=mobilenet_callbacks_ft,
    verbose=1
)

plot_training_curves(mobilenet_history_2, 'MobileNetV2 (Fine-Tuned)', 'figures/mobilenetv2_curves.png')

## 9. Model Evaluation

In [ ]:
def evaluate_model(model, test_gen, model_name, idx_to_class):
    """
    Evaluate model on test set.
    Returns: (y_true, y_pred, y_pred_prob, test_acc, test_loss)
    """
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    print(f'{model_name} — Test Loss: {loss:.4f} | Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')

    test_gen.reset()
    y_pred_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = test_gen.classes

    return y_true, y_pred, y_pred_prob, acc, loss


print('=' * 60)
print('MODEL EVALUATION ON TEST SET')
print('=' * 60)

cnn_true, cnn_pred, cnn_prob, cnn_acc, cnn_loss = evaluate_model(
    custom_cnn, test_generator, 'Custom CNN', idx_to_class)
resnet_true, resnet_pred, resnet_prob, resnet_acc, resnet_loss = evaluate_model(
    resnet_model, test_generator, 'ResNet50', idx_to_class)
mnet_true, mnet_pred, mnet_prob, mnet_acc, mnet_loss = evaluate_model(
    mobilenet_model, test_generator, 'MobileNetV2', idx_to_class)

In [ ]:
# ─── Confusion Matrices ───────────────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, model_name, save_path):
    """Plot normalised confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Raw Counts')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].tick_params(axis='x', rotation=30)

    # Normalised
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('Normalised (per true class)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')


classes_ordered = [idx_to_class[i] for i in range(NUM_CLASSES)]

plot_confusion_matrix(cnn_true,    cnn_pred,    classes_ordered, 'Custom CNN',  'figures/cm_custom_cnn.png')
plot_confusion_matrix(resnet_true, resnet_pred, classes_ordered, 'ResNet50',    'figures/cm_resnet50.png')
plot_confusion_matrix(mnet_true,   mnet_pred,   classes_ordered, 'MobileNetV2', 'figures/cm_mobilenetv2.png')

In [ ]:
# ─── Classification Reports ───────────────────────────────────────────────────
def print_classification_report(y_true, y_pred, class_names, model_name):
    print(f'\n=== Classification Report — {model_name} ===')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

print_classification_report(cnn_true,    cnn_pred,    classes_ordered, 'Custom CNN')
print_classification_report(resnet_true, resnet_pred, classes_ordered, 'ResNet50')
print_classification_report(mnet_true,   mnet_pred,   classes_ordered, 'MobileNetV2')

## 10. ROC Curves (One-vs-Rest)

In [ ]:
def plot_roc_curves(y_true, y_prob, class_names, model_name, save_path):
    """Plot ROC curves for each class using One-vs-Rest strategy."""
    y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    n_classes = y_true_bin.shape[1]

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        ax.plot(fpr[i], tpr[i], color=col, lw=1.8,
                label=f'{cls} (AUC = {roc_auc[i]:.3f})')
    ax.plot(all_fpr, mean_tpr, 'k--', lw=2,
            label=f'Macro Avg (AUC = {macro_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'gray', lw=1, linestyle=':')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {model_name}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')
    return macro_auc


cnn_auc    = plot_roc_curves(cnn_true,    cnn_prob,    classes_ordered, 'Custom CNN',  'figures/roc_custom_cnn.png')
resnet_auc = plot_roc_curves(resnet_true, resnet_prob, classes_ordered, 'ResNet50',    'figures/roc_resnet50.png')
mnet_auc   = plot_roc_curves(mnet_true,   mnet_prob,   classes_ordered, 'MobileNetV2', 'figures/roc_mobilenetv2.png')

## 11. Model Comparison Table

In [ ]:
# ─── Summary Comparison ───────────────────────────────────────────────────────
comparison_data = {
    'Model': ['Custom CNN', 'ResNet50 (Transfer)', 'MobileNetV2 (Transfer)'],
    'Test Accuracy': [f'{cnn_acc*100:.2f}%', f'{resnet_acc*100:.2f}%', f'{mnet_acc*100:.2f}%'],
    'Test Loss': [f'{cnn_loss:.4f}', f'{resnet_loss:.4f}', f'{mnet_loss:.4f}'],
    'Macro ROC-AUC': [f'{cnn_auc:.4f}', f'{resnet_auc:.4f}', f'{mnet_auc:.4f}'],
    'Parameters': [
        f"{custom_cnn.count_params():,}",
        f"{resnet_model.count_params():,}",
        f"{mobilenet_model.count_params():,}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print('\n=== Model Comparison Summary ===')
print(comparison_df.to_string(index=False))
comparison_df.to_csv('figures/model_comparison.csv', index=False)

# ─── Visual Comparison Bar Chart ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
models_list = ['Custom CNN', 'ResNet50', 'MobileNetV2']
accs = [cnn_acc * 100, resnet_acc * 100, mnet_acc * 100]
bars = ax.bar(models_list, accs, color=['steelblue', 'tomato', 'seagreen'], width=0.5)
ax.set_ylim(0, 105)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Model Accuracy Comparison — RAF-DB', fontweight='bold')
ax.axhline(y=max(accs), color='black', linestyle='--', alpha=0.4)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/model_comparison_bar.png')

## 12. Grad-CAM Visualisation (Best Model)

In [ ]:
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap for the specified convolutional layer."""
    # Model up to the last conv layer + classification head
    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_path, model, last_conv_layer_name, class_names, img_size=(224, 224)):
    """Load image, compute Grad-CAM, overlay on original image."""
    img = keras.preprocessing.image.load_img(img_path, target_size=img_size)
    img_array = keras.preprocessing.image.img_to_array(img) / 255.0
    img_array_exp = np.expand_dims(img_array, axis=0)

    heatmap = make_gradcam_heatmap(img_array_exp, model, last_conv_layer_name)
    pred_class = np.argmax(model.predict(img_array_exp, verbose=0))

    # Resize heatmap to image size
    heatmap_resized = cv2.resize(heatmap, img_size)
    heatmap_colored = cv2.applyColorMap(
        (heatmap_resized * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = (heatmap_colored / 255.0) * 0.4 + img_array
    superimposed = np.clip(superimposed, 0, 1)
    return img_array, superimposed, class_names[pred_class]


# Find last conv layer name in Custom CNN
last_conv_name = None
for layer in reversed(custom_cnn.layers):
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name
        break
print(f'Last conv layer for Grad-CAM: {last_conv_name}')

# Sample one image per class for Grad-CAM
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(20, 6))
fig.suptitle('Grad-CAM Visualisations — Custom CNN', fontsize=13, fontweight='bold')

for col, cls in enumerate(sorted(class_indices.keys())):
    cls_path = os.path.join(TEST_DIR, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        img_path = os.path.join(cls_path, imgs[0])
        try:
            orig, cam_overlay, pred = overlay_gradcam(
                img_path, custom_cnn, last_conv_name, classes_ordered
            )
            axes[0, col].imshow(orig)
            axes[0, col].set_title(f'True: {cls}', fontsize=8)
            axes[0, col].axis('off')
            axes[1, col].imshow(cam_overlay)
            axes[1, col].set_title(f'Pred: {pred}', fontsize=8,
                                    color='green' if pred == cls else 'red')
            axes[1, col].axis('off')
        except Exception as e:
            axes[0, col].axis('off')
            axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('figures/gradcam_visualisations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/gradcam_visualisations.png')

## 13. Error Analysis — Misclassified Examples

In [ ]:
# ─── Show Hard Examples (Misclassified) ───────────────────────────────────────
def show_misclassified(model, test_gen, idx_to_class, n_samples=12, model_name='Model'):
    """Display a grid of misclassified examples."""
    test_gen.reset()
    filenames  = test_gen.filenames
    true_labels = test_gen.classes
    test_gen.reset()
    pred_probs = model.predict(test_gen, verbose=0)
    pred_labels = np.argmax(pred_probs, axis=1)

    misclassified_idx = np.where(pred_labels != true_labels)[0]
    sample_idx = np.random.choice(misclassified_idx,
                                   min(n_samples, len(misclassified_idx)),
                                   replace=False)

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    fig.suptitle(f'Misclassified Examples — {model_name}\n'
                 f'(Green = True, Red = Predicted)', fontsize=12, fontweight='bold')
    axes = axes.flatten()

    for i, idx in enumerate(sample_idx[:12]):
        img_path = os.path.join(TEST_DIR, filenames[idx])
        img = plt.imread(img_path)
        axes[i].imshow(img)
        true_cls = idx_to_class[true_labels[idx]]
        pred_cls = idx_to_class[pred_labels[idx]]
        axes[i].set_title(f'T: {true_cls}\nP: {pred_cls}', fontsize=8,
                          color='darkred')
        axes[i].axis('off')

    for j in range(i + 1, 12):
        axes[j].axis('off')

    plt.tight_layout()
    fname = f'figures/misclassified_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fname}')
    print(f'Total misclassified: {len(misclassified_idx)} / {len(true_labels)} '
          f'({len(misclassified_idx)/len(true_labels)*100:.1f}%)')


show_misclassified(custom_cnn, test_generator, idx_to_class, model_name='Custom CNN')
show_misclassified(resnet_model, test_generator, idx_to_class, model_name='ResNet50')

## 14. Save Models and Final Summary

In [ ]:
# ─── Save All Models ──────────────────────────────────────────────────────────
custom_cnn.save('model_custom_cnn_final.keras')
resnet_model.save('model_resnet50_final.keras')
mobilenet_model.save('model_mobilenetv2_final.keras')

print('All models saved.')

# ─── Final Summary ────────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('FINAL SUMMARY — RAF-DB Facial Expression Recognition')
print('=' * 65)
print(comparison_df.to_string(index=False))
print('\nGenerated Figures:')
for f in sorted(os.listdir('figures')):
    print(f'  figures/{f}')
print('\nProject complete. Author: Bhanu Prasad Mudraboina')